# Association Rule Mining C1 — Streaming Series

## Role 1: Setup, EDA and Transaction Building

This section prepares the streaming series dataset for Association Rule Mining.
It covers initial exploratory data analysis, construction of viewing-session
transactions, basket-size analysis, and creation of a separate show lookup table.

In [ ]:
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/streaming_series_watched.csv")

df.head()

## 1. Exploratory Data Analysis (EDA)

The first step is to inspect the structure, dimensions, data types, missing values, and basic characteristics of the streaming-series dataset.

In [ ]:
# Dataset dimensions
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types and non-null counts
print("\nDataset Information:")
df.info()

In [ ]:
# Missing values
print("Missing values per column:")
display(df.isnull().sum())

# Duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())

In [ ]:
print(df.columns.tolist())

In [ ]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

In [ ]:
df = df.rename(columns={
    "ls dataviewing_session_id": "viewing_session_id"
})

print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

print("Total records:", len(df))
print("Unique viewing sessions:", df["viewing_session_id"].nunique())
print("Unique shows:", df["show_id"].nunique())
print("Unique item names:", df["item_name"].nunique())
print("Unique transaction dates:", df["transaction_date"].nunique())

print("\nEarliest transaction date:", df["transaction_date"].min())
print("Latest transaction date:", df["transaction_date"].max())

In [ ]:
basket_sizes = df.groupby("viewing_session_id")["show_id"].nunique()

basket_sizes.describe()

### Basket Size Distribution

A basket represents the unique shows watched during one viewing session.
The basket-size distribution shows how many sessions contain one, two, three, or more unique shows.

In [ ]:
basket_size_distribution = (
    basket_sizes
    .value_counts()
    .sort_index()
    .rename_axis("basket_size")
    .reset_index(name="number_of_sessions")
)

basket_size_distribution

In [ ]:
%pip install --upgrade --force-reinstall matplotlib
import matplotlib.pyplot as plt

basket_size_distribution.plot(
    x="basket_size",
    y="number_of_sessions",
    kind="bar",
    legend=False
)

plt.title("Basket Size Distribution")
plt.xlabel("Number of Unique Shows per Session")
plt.ylabel("Number of Sessions")
plt.xticks(rotation=0)
plt.show()

## 2. Show Lookup Table

A separate lookup table is created to retain descriptive information about each show.
This allows the transaction baskets to contain only show identifiers while show names,
categories, and average episode runtimes remain available for interpretation.

In [ ]:
show_lookup = (
    df[
        ["show_id", "item_name", "category", "avg_episode_minutes"]
    ]
    .drop_duplicates()
    .sort_values("show_id")
    .reset_index(drop=True)
)

show_lookup

In [ ]:
print("Number of unique shows:", df["show_id"].nunique())
print("Rows in lookup table:", len(show_lookup))
print("Duplicate show IDs:", show_lookup["show_id"].duplicated().sum())

## 3. Building Transactions

For Association Rule Mining, the dataset is transformed from individual viewing records
into transaction baskets. Each `viewing_session_id` represents one transaction, and the
shows watched during that session form the items in the basket.

`transaction_date` is not included in the baskets because the association analysis focuses
on combinations of shows watched within the same session.

In [ ]:
transactions = (
    df.groupby("viewing_session_id")["show_id"]
      .apply(lambda x: sorted(x.unique()))
      .reset_index(name="basket")
)

transactions.head(10)

In [ ]:
["viewing_session_id", "show_id"]

In [ ]:
print("Original viewing records:", len(df))
print("Number of transaction baskets:", len(transactions))
print("Unique viewing sessions:", df["viewing_session_id"].nunique())

transactions.head()

## 4. Transaction Validation

The prepared transactions are validated to ensure that each viewing session corresponds
to exactly one basket and that no transaction dates or descriptive attributes are included
in the baskets.

In [ ]:
print("Unique viewing sessions:", df["viewing_session_id"].nunique())
print("Transaction baskets:", len(transactions))
print("Unique shows:", df["show_id"].nunique())
print("Lookup table rows:", len(show_lookup))

assert len(transactions) == df["viewing_session_id"].nunique()
assert len(show_lookup) == df["show_id"].nunique()

print("\nValidation successful.")

In [ ]:
show_lookup.to_csv("../outputs/show_lookup.csv", index=False)

In [ ]:
transactions_export = transactions.copy()

transactions_export["basket"] = transactions_export["basket"].apply(
    lambda items: ",".join(items)
)

transactions_export.to_csv(
    "../outputs/session_transactions.csv",
    index=False
)

print("Files saved successfully.")

In [ ]:
import os

print(os.listdir("../outputs"))

## Role 1 Summary

The streaming-series dataset was successfully loaded and prepared for Association Rule Mining.
Initial EDA was conducted to examine the dataset dimensions, viewing sessions, unique shows,
transaction dates, missing values, duplicates, and basket-size distribution.

The data was then transformed into transaction baskets by grouping records according to
`viewing_session_id`, with each session representing one basket of unique `show_id` values.
`transaction_date` was excluded from the transaction baskets because it is not required for
the association analysis.

A separate show lookup table was also created containing `show_id`, `item_name`, `category`,
and `avg_episode_minutes`. The prepared transaction and lookup datasets were saved for use in
the subsequent Association Rule Mining stages.

## Role 2: Choosing `min_support` and Generating Frequent Itemsets

In this section we build on the transaction baskets prepared by Member 1. The baskets are converted into a one-hot encoded format so that they can be used for association rule mining.

We then select a suitable `min_support` value based on the actual number of viewing sessions in our dataset. This helps ensure that the threshold is appropriate for our data rather than being chosen randomly.

Finally, we use the selected threshold to generate frequent itemsets and perform a basic check of the results. The resulting itemsets will then be passed to Member 3 for association rule generation and redundancy filtering.

In [ ]:
%pip install mlxtend

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori



## 2. Preparing baskets for itemset mining

Member 1's output (`session_transactions.csv`) stores each session's basket as a
comma-separated string, since lists can't be written directly to CSV. The first
step here is loading that file back in and restoring the basket column to an
actual Python list, so it can be one-hot encoded.

In [ ]:
# --- Step 1: Load Member 1's transaction output ---
transactions = pd.read_csv("../outputs/session_transactions.csv")

# basket was flattened to a comma-separated string for CSV export — restore it to a list
transactions["basket"] = transactions["basket"].str.split(",")

print("Number of transaction baskets:", len(transactions))
transactions.head()

In [ ]:
# --- Step 2: One-hot encode the baskets ---
# apriori() requires a wide True/False matrix, one column per item — not the
# list-of-lists format the baskets are currently in.
basket_list = transactions["basket"].tolist()

te = TransactionEncoder()
te_array = te.fit(basket_list).transform(basket_list)

onehot = pd.DataFrame(te_array, columns=te.columns_)

print("One-hot matrix shape:", onehot.shape)
onehot.head()

## 3. Individual item support

Before choosing `min_support`, it helps to see how often each show is actually
watched across the 3500 sessions. This grounds the threshold decision in the
real distribution for this dataset, rather than guessing a round number.

In [ ]:
# --- Step 3: Individual item support, translated into session counts ---
n_sessions = len(transactions)

item_support = onehot.mean().sort_values(ascending=False)

item_support_df = item_support.reset_index()
item_support_df.columns = ["show_id", "support"]
item_support_df["num_sessions"] = (item_support_df["support"] * n_sessions).round().astype(int)

item_support_df

In [ ]:
# --- Step 4: Trial a few candidate min_support values before committing ---
# With only 15 items, a threshold that's too low will make apriori's itemset
# count explode (too many combinations survive); too high and almost nothing
# will qualify. Trying several values first is how the "chosen from the
# observed distribution" part of Criterion 3 gets satisfied.
candidate_thresholds = [0.01, 0.02, 0.03, 0.05]

for min_sup in candidate_thresholds:
    itemsets = apriori(onehot, min_support=min_sup, use_colnames=True)
    required_sessions = round(min_sup * n_sessions)
    print(
        f"min_support={min_sup:<5} "
        f"(>= {required_sessions} of {n_sessions} sessions) "
        f"-> {len(itemsets)} itemsets"
    )


## 4. Final min_support decision

### Choosing the Minimum Support Threshold

Several candidate `min_support` values were tested against the 3,500 viewing sessions:

- `0.01` → 94 frequent itemsets (at least 35 sessions)
- `0.02` → 31 frequent itemsets (at least 70 sessions)
- `0.03` → 30 frequent itemsets (at least 105 sessions)
- `0.05` → 22 frequent itemsets (at least 175 sessions)

The threshold of `0.01` produced a much larger number of itemsets, while increasing the threshold beyond `0.02` resulted in relatively small reductions. Therefore `0.02` was selected as a reasonable balance between retaining recurring co-viewing patterns and removing very low-support patterns. This means an itemset must occur in at least 70 of the 3,500 viewing sessions to be considered frequent.

In [ ]:
# --- Step 5: Generate final frequent itemsets at the chosen threshold ---
FINAL_MIN_SUPPORT = 0.02   # <- replace with whichever value you justified above

frequent_itemsets = apriori(onehot, min_support=FINAL_MIN_SUPPORT, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)


In [ ]:
# --- Step 6: Sanity check on itemset sizes ---
# Confirms the mined itemsets aren't all singletons (which would mean the
# threshold is too high to find any real co-viewing pairs/triples).
frequent_itemsets["itemset_size"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets["itemset_size"].value_counts().sort_index()

In [ ]:
# --- Step 7: Save output for Member 3 (thresholds & redundancy removal) ---
frequent_itemsets.to_pickle("../outputs/frequent_itemsets.pkl")

print("Frequent itemsets saved.")

## Role 2 Summary

In this section Member 1's session transaction output was loaded and the viewing baskets were restored into lists of shows. The baskets were then converted into a one-hot encoded transaction matrix using `TransactionEncoder` where each row represents a viewing session and each column represents a show.

Several candidate `min_support` values were tested using the 3,500 viewing sessions. The results were 94 frequent itemsets at 0.01, 31 at 0.02, 30 at 0.03, and 22 at 0.05. Based on the observed distribution, `min_support = 0.02` was selected as a reasonable threshold. This requires an itemset to appear in at least 70 of the 3,500 sessions to be considered frequent.

Finally, the frequent itemsets were generated using the Apriori algorithm, sorted by support, checked by itemset size, and saved as `frequent_itemsets.pkl` for use in the next stage of the analysis.


## Role 3: Confidence/Lift Thresholds and Redundancy Removal

This section generates association rules from the frequent itemsets produced by Member 2.

Before selecting confidence and lift thresholds, the distributions of these metrics are
examined so that the cutoffs are based on the actual rules produced by the dataset rather
than being selected arbitrarily.

After selecting suitable thresholds, redundant rules are removed. This includes
subset-redundant rules and bidirectional duplicate rules. The number of rules before
and after filtering is reported to show the effect of the cleanup process.

In [ ]:
#Loading itemsets and generating the intial rules

# Import association rule function
import pandas as pd
from mlxtend.frequent_patterns import association_rules

# Load frequent itemsets produced by Member 2
frequent_itemsets = pd.read_pickle("../outputs/frequent_itemsets.pkl")

print("Number of frequent itemsets:", len(frequent_itemsets))

# Generate all possible rules first.
# A very low confidence threshold is used here because we want to inspect
# the confidence and lift distributions BEFORE choosing the final cutoffs.
rules_all = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.0
)

print("Number of association rules generated:", len(rules_all))

rules_all.head()



In [ ]:
# Examine the distribution of confidence and lift before selecting thresholds
rules_all[["confidence", "lift"]].describe()


In [ ]:
# Examine useful percentiles to understand the distribution more clearly
rules_all[["confidence", "lift"]].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95]
)

In [ ]:
import matplotlib.pyplot as plt

rules_all["confidence"].hist(bins=20)

plt.title("Distribution of Rule Confidence")
plt.xlabel("Confidence")
plt.ylabel("Number of Rules")
plt.show()

In [ ]:
rules_all["lift"].hist(bins=20)

plt.title("Distribution of Rule Lift")
plt.xlabel("Lift")
plt.ylabel("Number of Rules")
plt.show()

In [ ]:
# Inspect the strongest rules

rules_all[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].sort_values(
    ["confidence", "lift"],
    ascending=False
).head(20)

In [ ]:
# 0.30 sits between the 25th percentile (0.265) and median (0.380)
# It gives a reasonable balance: rules with very low confidence are removed
MIN_CONFIDENCE = 0.30
# minimum lift of 2 because the observed lift values were already high,
# ranging from approximately 1.92 to 5.24,
# and a lift of 2 represents a strong positive association.
MIN_LIFT = 2.00

#Apply the selected confidence and lift thresholds to filter the rules
rules_filtered = rules_all[
    (rules_all["confidence"] >= MIN_CONFIDENCE) &
    (rules_all["lift"] >= MIN_LIFT)
].copy().reset_index(drop=True)

print("Rules before threshold filtering:", len(rules_all))
print("Rules after threshold filtering:", len(rules_filtered))
print("Rules removed:", len(rules_all) - len(rules_filtered))



### Confidence and Lift Threshold Selection

The confidence and lift distributions were examined before selecting the final
thresholds. This ensured that the cutoffs were based on the rules generated from
the dataset rather than being chosen arbitrarily.

A minimum confidence of **0.30** was selected to retain rules where the
consequent occurs reasonably often when the antecedent occurs.

A minimum lift of **2.00** was selected to retain positive associations.
A lift greater than 1 indicates that the antecedent and consequent occur together
more often than would be expected if they were independent.

## Redundancy Removal

In [ ]:
rules_clean = rules_filtered.copy()

print("Rules before redundancy removal:", len(rules_clean))

### 1. Remove bidirectional duplicates

In [ ]:
# Remove bidirectional duplicate rules
# A canonical key treats A -> B and B -> A as the same item relationship.

def bidirectional_key(row):
    left = tuple(sorted(row["antecedents"]))
    right = tuple(sorted(row["consequents"]))

    return tuple(sorted([left, right]))

rules_clean["direction_key"] = rules_clean.apply(
    bidirectional_key,
    axis=1
)

# Keep the stronger rule based first on confidence and then lift
rules_clean = (
    rules_clean
    .sort_values(
        ["confidence", "lift"],
        ascending=False
    )
    .drop_duplicates(
        subset="direction_key",
        keep="first"
    )
    .reset_index(drop=True)
)

print("Rules after bidirectional duplicate removal:", len(rules_clean))

### 2. Remove Subset-redundant rules

In [ ]:
def remove_subset_redundant_rules(rules):
    """
    Remove a rule when a simpler antecedent predicts the same consequent
    with equal or better confidence.
    """

    keep = []

    for i, rule in rules.iterrows():
        redundant = False

        for j, simpler_rule in rules.iterrows():

            if i == j:
                continue

            # Consequents must be the same
            same_consequent = (
                rule["consequents"] == simpler_rule["consequents"]
            )

            # Simpler antecedent must be a proper subset
            simpler_antecedent = (
                simpler_rule["antecedents"] < rule["antecedents"]
            )

            # Simpler rule must perform at least as well
            equally_or_more_confident = (
                simpler_rule["confidence"] >= rule["confidence"]
            )

            if (
                same_consequent
                and simpler_antecedent
                and equally_or_more_confident
            ):
                redundant = True
                break

        keep.append(not redundant)

    return rules.loc[keep].reset_index(drop=True)


before_subset = len(rules_clean)

rules_clean = remove_subset_redundant_rules(rules_clean)

after_subset = len(rules_clean)

print("Rules before subset redundancy removal:", before_subset)
print("Rules after subset redundancy removal:", after_subset)
print("Subset-redundant rules removed:", before_subset - after_subset)

In [ ]:
# Remove temporary helper column
rules_clean = rules_clean.drop(
    columns=["direction_key"],
    errors="ignore"
)

# Rank the remaining rules
rules_clean = rules_clean.sort_values(
    ["confidence", "lift"],
    ascending=False
).reset_index(drop=True)

rules_clean[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].head(20)

In [ ]:
print("Initial association rules:", len(rules_all))
print("After confidence/lift thresholds:", len(rules_filtered))
print("After redundancy removal:", len(rules_clean))
print(
    "Total rules removed:",
    len(rules_all) - len(rules_clean)
)

In [ ]:
rules_clean.to_pickle("../outputs/clean_association_rules.pkl")
rules_clean.to_csv(
    "../outputs/clean_association_rules.csv",
    index=False
)

print("Clean association rules saved successfully.")

## Role 3 Summary

Association rules were generated from the frequent itemsets produced using the minimum support threshold selected by Member 2. Rather than selecting confidence and lift thresholds arbitrarily, their distributions and
percentiles were first examined.

Based on the observed distributions, a minimum confidence of **[0.30]** and a
minimum lift of **[2.00]** were selected. Confidence measures how reliably the consequent occurs when the antecedent occurs, while lift measures the strength of the relationship compared with what would be expected if the items were independent

The threshold filtering reduced the rules from **[44]** to
**[29]**. Redundancy cleanup was then performed by removing
bidirectional duplicates and subset-redundant rules. For bidirectional pairs,
the stronger rule was retained based on confidence and lift.

After redundancy removal, **[21]** rules remained. The cleaned rules
were then saved for Member 4 to perform show-name translation, interpretation, and temporal validation.

## Role 4: Rule Interpretation and Temporal Validation

This section takes the 21 cleaned rules from Member 3 and does two things.

**Rule interpretation.** The rules were mined on `show_id` values, so they are translated back into
show names and genres using Member 1's lookup table. We then ask which rules are actually
interesting, as opposed to merely having a high confidence or lift.

**Temporal validation.** Every rule so far was mined and judged on the same data. Here the
sessions are split by `transaction_date` into an earlier and a later period. Rules are mined on the
earlier period only, and their confidence is then rechecked on the later period, which the mining
never saw. A rule that only looked strong because of a quirk of one time window would fail this check.

In [ ]:
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

rules_clean = pd.read_pickle("../outputs/clean_association_rules.pkl")
show_lookup = pd.read_csv("../outputs/show_lookup.csv")

raw = pd.read_csv("../data/streaming_series_watched.csv")
raw.columns = raw.columns.str.strip().str.replace(
    "ls dataviewing_session_id", "viewing_session_id", regex=False
)
raw["transaction_date"] = pd.to_datetime(raw["transaction_date"])

print("Cleaned rules loaded:", len(rules_clean))
print("Shows in lookup table:", len(show_lookup))
print("Viewing sessions:", raw["viewing_session_id"].nunique())

### 1. Translate Rules into Show Names

Mining was done on identifiers (`S01`, `S02`, ...) and names are only attached at the reporting stage.
Each rule is also tagged with the genre(s) it spans, and as Same genre or Cross genre. That tag is
used in the next section.

In [ ]:
# Lookups: show_id -> name and show_id -> genre
name_of = show_lookup.set_index("show_id")["item_name"]
genre_of = show_lookup.set_index("show_id")["category"]
N_SESSIONS = raw["viewing_session_id"].nunique()


def names(show_ids):
    """frozenset of show_ids -> 'Show A + Show B' (sorted for stable output)."""
    return " + ".join(sorted(name_of[s] for s in show_ids))


def genres(show_ids):
    """frozenset of show_ids -> sorted list of the distinct genres involved."""
    return sorted({genre_of[s] for s in show_ids})


readable = rules_clean.copy()
readable["antecedent_shows"] = readable["antecedents"].apply(names)
readable["consequent_shows"] = readable["consequents"].apply(names)
readable["rule"] = readable["antecedent_shows"] + "  →  " + readable["consequent_shows"]
readable["sessions"] = (readable["support"] * N_SESSIONS).round().astype(int)

# Genre(s) spanned by the whole rule (antecedent + consequent)
readable["genre_list"] = readable.apply(lambda r: genres(r["antecedents"] | r["consequents"]), axis=1)
readable["genres"] = readable["genre_list"].apply(" / ".join)
readable["rule_type"] = np.where(readable["genre_list"].apply(len) == 1, "Same genre", "Cross genre")

readable[["rule", "genres", "rule_type", "sessions", "support", "confidence", "lift"]].round(3)

In [ ]:
def explain(row):
    return (
        f"If a session includes {row['antecedent_shows']}, then in {row['confidence']:.0%} of such sessions "
        f"{row['consequent_shows']} is also watched. That is {row['lift']:.1f}x more often than chance "
        f"(seen together in {row['sessions']} of {N_SESSIONS} sessions)."
    )

for _, row in readable.head(5).iterrows():
    print("•", explain(row), "\n")

2. High confidence and high lift do not automatically make a rule useful. A rule is only useful if it tells
us something we could not already read off the show catalogue. The tables below separate the rules that
merely restate the genre labels from the ones that go beyond them.

In [ ]:
# Same-genre rules are the "expected" ones; cross-genre rules are the surprising ones
type_summary = readable.groupby("rule_type").agg(
    n_rules=("rule", "count"),
    mean_confidence=("confidence", "mean"),
    mean_lift=("lift", "mean"),
).round(3)
display(type_summary)

print("Rules per genre combination:")
display(readable["genres"].value_counts().rename("n_rules").to_frame())

In [ ]:
# Cross-genre rules: these tell us something the show catalogue does not already say.
# "Uplift over baseline" = confidence minus how often the consequent is watched anyway.
cross = readable[readable["rule_type"] == "Cross genre"].copy()
cross["baseline"] = cross["consequent support"]
cross["uplift_over_baseline"] = cross["confidence"] - cross["baseline"]

cross[["rule", "genres", "sessions", "confidence", "baseline", "uplift_over_baseline", "lift"]].round(3)

#### Discussion

**1. Most rules restate the catalogue.** 19 of the 21 rules stay inside one genre (Crime Drama, Sci-Fi,
Reality TV, Sitcom, Kids). They have the highest lifts (mean 3.55), but a plain "more from the same genre"
shelf would already make the same recommendations. High lift here is not the same as high insight.

**2. The 21 rules are really 15 relationships.** Each trio of shows that is watched together produces three
rules from a single itemset: Crime City + The Detective + Silent Witness Redux (120 sessions), Space Frontier +
Galactic Wars + Time Anomaly (106) and Love Island Nairobi + Real Life Diaries + Cooking Masters (102). That is 9 rules describing 3
behaviours, plus 12 pair rules.

**3. Confidence needs the baseline next to it.** Crime City is watched in 22.5% of all sessions and Toon Town in
22.3%, so a rule ending in either one gets a fair amount of confidence for free. Conversely, the rule
ending in Silent Witness Redux (only 10% of sessions) has just 0.385 confidence but a lift of 3.8. The "uplift over
baseline" column shows the honest gain: the two cross-genre rules add about 33 to 37 percentage points.

**4. The two cross-genre rules are the interesting ones.**
- **True Crime Files (Documentary) - Crime City (Crime Drama)**: 60% confidence against a 22.5% baseline. Viewers treat the
  documentary as part of the crime family even though the catalogue files it under a different category.
- **Parenting Unscripted (Reality TV) - Toon Town (Kids)**: 55.7% against a 22.3% baseline. Parenting Unscripted appears in no rule with
  the other Reality TV shows, so behaviourally it sits with the Kids content. A plausible explanation is family or
  parent-and-child viewing, but that is a hypothesis: the data has no user or age information to confirm it.

**5. What is missing is informative too.** No rule connects Crime, Sci-Fi, Reality TV and Sitcom to each other at these
thresholds. Audiences appear to be strongly segmented by genre, so the recommender should not expect much
cross-genre signal beyond the two cases above.

**6. Direction is not meaningful for pairs.** Member 3 kept only the stronger direction of each A-B / B-A pair, but the two
directions are often nearly tied (Laugh Track - Office Antics: 0.615 vs 0.610). Those pairs should be read as
"watched together", not A leads to B.

### 3. Temporal Validation on a Held-Out Period

**Method.**
1. Attach `transaction_date` to each session and split chronologically at **1 April 2025**, which gives an earlier period
   (Jan to Mar, 1,778 sessions) and a later period (Apr to Jun, 1,722 sessions). The split is by date, and each session
   sits on exactly one date, so no session is cut in half. The halves are balanced so that even the three-show rules
   rest on a reasonable number of held-out sessions.
2. Re-run Members 2 and 3's full pipeline (same `min_support`, confidence and lift thresholds, same redundancy
   removal) on the **earlier period only**.
3. Recompute each rule's confidence on the **later period** and compare.

Re-mining on the earlier period matters: Member 3's 21 rules were selected using all the data, so simply
recomputing those on two halves would let the later period influence which rules were chosen.

In [ ]:
# Each session must fall on exactly one date, otherwise splitting by date would cut sessions in half
session_dates = raw.groupby("viewing_session_id")["transaction_date"].agg(["min", "max"])
assert (session_dates["min"] == session_dates["max"]).all(), "A session spans several dates!"
session_dates = session_dates["min"].rename("transaction_date")

# Rebuild baskets (Member 1's file) and attach each session's date
transactions = pd.read_csv("../outputs/session_transactions.csv")
transactions["basket"] = transactions["basket"].str.split(",")
transactions = transactions.merge(session_dates, on="viewing_session_id", how="left")

print("Sessions without a date:", transactions["transaction_date"].isna().sum())
print("Date range:", transactions["transaction_date"].min().date(), "to", transactions["transaction_date"].max().date())
print()
print("Sessions per month:")
print(transactions.groupby(transactions["transaction_date"].dt.to_period("M")).size().to_string())

In [ ]:
# Chronological split: earlier period = "training" (rules are mined here),
# later period = held-out (rules are only checked here, never mined).
CUTOFF_DATE = pd.Timestamp("2025-04-01")

earlier = transactions[transactions["transaction_date"] < CUTOFF_DATE]
later = transactions[transactions["transaction_date"] >= CUTOFF_DATE]

print(f"Earlier period: {earlier['transaction_date'].min().date()} to {earlier['transaction_date'].max().date()} "
      f"-> {len(earlier)} sessions ({len(earlier) / len(transactions):.0%})")
print(f"Later period:   {later['transaction_date'].min().date()} to {later['transaction_date'].max().date()} "
      f"-> {len(later)} sessions ({len(later) / len(transactions):.0%})")

# No session can appear on both sides
assert set(earlier["viewing_session_id"]).isdisjoint(later["viewing_session_id"])

In [ ]:
# Same thresholds that Members 2 and 3 justified on the full data
MIN_SUPPORT = 0.02
MIN_CONFIDENCE = 0.30
MIN_LIFT = 2.00
ALL_SHOWS = sorted(show_lookup["show_id"])


def to_onehot(baskets):
    """Baskets -> one-hot DataFrame with a column for EVERY show (so held-out data never lacks a column)."""
    enc = TransactionEncoder().fit([ALL_SHOWS])
    return pd.DataFrame(enc.transform(baskets), columns=enc.columns_)


def mine_rules(baskets):
    """Members 2 + 3's pipeline as one function: Apriori -> confidence/lift filter ->
    bidirectional duplicate removal -> subset-redundancy removal."""
    onehot = to_onehot(baskets)
    freq = apriori(onehot, min_support=MIN_SUPPORT, use_colnames=True)
    r = association_rules(freq, metric="confidence", min_threshold=0.0)
    r = r[(r["confidence"] >= MIN_CONFIDENCE) & (r["lift"] >= MIN_LIFT)].copy()

    # 1. keep only the stronger direction of each A->B / B->A pair
    r["pair"] = r.apply(lambda x: frozenset([x["antecedents"], x["consequents"]]), axis=1)
    r = r.sort_values(["confidence", "lift"], ascending=False).drop_duplicates("pair")

    # 2. drop a rule if a simpler antecedent gives the same consequent with >= confidence
    rows = list(r.itertuples())
    keep = [
        not any(o.consequents == x.consequents and o.antecedents < x.antecedents
                and o.confidence >= x.confidence for o in rows)
        for x in rows
    ]
    return r.loc[keep].drop(columns="pair").sort_values(["confidence", "lift"], ascending=False).reset_index(drop=True)


def rule_keys(rules_df):
    return {(r.antecedents, r.consequents) for r in rules_df.itertuples()}


# Sanity check: on the FULL data the function must reproduce Member 3's 21 rules exactly
full_rules = mine_rules(transactions["basket"].tolist())
print("Rules from full data:", len(full_rules))
print("Identical to Member 3's cleaned rules:", rule_keys(full_rules) == rule_keys(rules_clean))

In [ ]:
# Mine ONLY on the earlier period
train_rules = mine_rules(earlier["basket"].tolist())
print(f"Rules mined on the earlier period alone: {len(train_rules)}")

# Overlap with the rules Member 3 mined on the whole dataset
same = rule_keys(train_rules) & rule_keys(rules_clean)
only_full = rule_keys(rules_clean) - rule_keys(train_rules)
only_early = rule_keys(train_rules) - rule_keys(rules_clean)
print(f"Also in Member 3's full-data rules: {len(same)}")
print(f"Full-data rules NOT found in the earlier period: {len(only_full)}")
print(f"Earlier-period rules NOT in the full-data set:   {len(only_early)}")

print("\nFull-data only:")
for a, c in only_full:
    print("  ", names(a), "->", names(c))
print("Earlier-period only:")
for a, c in only_early:
    print("  ", names(a), "->", names(c))

#### Rechecking confidence on the held-out period

A change of more than 0.15 in confidence is treated as a meaningful shift, following the lecturer's lab. Two extra checks
are added: a z-score (how many standard errors the later confidence sits from the earlier one, given the number of
later sessions containing the antecedent) and whether the lift is still at least 2.0.

In [ ]:
# Recheck each earlier-period rule on the held-out (later) period
later_onehot = to_onehot(later["basket"].tolist())


def measure(onehot, antecedent, consequent):
    """Confidence, lift and antecedent count of one specific rule on a given period."""
    a = onehot[list(antecedent)].all(axis=1)
    c = onehot[list(consequent)].all(axis=1)
    n_ante = int(a.sum())
    conf = (a & c).sum() / n_ante if n_ante else np.nan
    lift = conf / c.mean() if n_ante and c.mean() > 0 else np.nan
    return conf, lift, n_ante


LARGE_DROP = 0.15   # same "meaningful shift" threshold used in the lecturer's lab

results = []
for r in train_rules.itertuples():
    conf_late, lift_late, n_ante_late = measure(later_onehot, r.antecedents, r.consequents)
    # Standard error of a proportion under "the training confidence is the truth"
    se = np.sqrt(r.confidence * (1 - r.confidence) / n_ante_late) if n_ante_late else np.nan
    results.append({
        "rule": f"{names(r.antecedents)}  →  {names(r.consequents)}",
        "train_confidence": r.confidence,
        "later_confidence": conf_late,
        "change": conf_late - r.confidence,
        "z_score": (conf_late - r.confidence) / se if se else np.nan,
        "train_lift": r.lift,
        "later_lift": lift_late,
        "later_antecedent_sessions": n_ante_late,
    })

validation = pd.DataFrame(results)
validation["status"] = np.where(validation["change"].abs() > LARGE_DROP, "Shifted", "Stable")
validation["lift_holds"] = validation["later_lift"] >= MIN_LIFT
validation = validation.sort_values("change").reset_index(drop=True)
validation.round(3)

In [ ]:
n_rules = len(validation)
n_stable = (validation["status"] == "Stable").sum()
print(f"Rules checked on the later period:                 {n_rules}")
print(f"Stable (|change in confidence| <= {LARGE_DROP}):            {n_stable}")
print(f"Shifted (|change| > {LARGE_DROP}):                       {n_rules - n_stable}")
print(f"Rules whose lift is still >= {MIN_LIFT}:                  {validation['lift_holds'].sum()}")
print(f"Rules within +/- 2 standard errors (pure noise):   {(validation['z_score'].abs() <= 2).sum()}")
print()
print(f"Mean confidence, earlier period : {validation['train_confidence'].mean():.3f}")
print(f"Mean confidence, later period   : {validation['later_confidence'].mean():.3f}")
print(f"Mean change                     : {validation['change'].mean():+.3f}")
print(f"Largest drop / largest rise     : {validation['change'].min():+.3f} / {validation['change'].max():+.3f}")
print(f"Smallest antecedent count in later period: {validation['later_antecedent_sessions'].min()} sessions")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: earlier vs later confidence. Points on the diagonal did not change.
ax = axes[0]
ax.scatter(validation["train_confidence"], validation["later_confidence"], s=60, edgecolor="k", alpha=0.85)
lims = [0.2, 0.95]
ax.plot(lims, lims, "k--", lw=1, label="No change")
ax.fill_between(lims, [l - LARGE_DROP for l in lims], [l + LARGE_DROP for l in lims],
                color="grey", alpha=0.12, label=f"±{LARGE_DROP} band")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Confidence, earlier period (mined)")
ax.set_ylabel("Confidence, later period (held out)")
ax.set_title("Rule confidence: earlier vs later")
ax.legend(loc="upper left")

# Right: side-by-side bars per rule
ax = axes[1]
plot_df = validation.sort_values("train_confidence")
y = np.arange(len(plot_df))
ax.barh(y - 0.2, plot_df["train_confidence"], height=0.4, label="Earlier", color="#4F6EA4")
ax.barh(y + 0.2, plot_df["later_confidence"], height=0.4, label="Later", color="#E08E45")
ax.set_yticks(y)
ax.set_yticklabels(plot_df["rule"], fontsize=7)
ax.set_xlabel("Confidence")
ax.set_title("Confidence per rule in each period")
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# The 21 rules Member 3 delivered were mined on ALL the data, so their "earlier" numbers are not
# a fair test on their own. Still, comparing both halves shows which ones are consistent everywhere.
rows = []
early_onehot = to_onehot(earlier["basket"].tolist())
for r in readable.itertuples():
    c1, _, n1 = measure(early_onehot, r.antecedents, r.consequents)
    c2, _, n2 = measure(later_onehot, r.antecedents, r.consequents)
    rows.append({"rule": r.rule, "full_data": r.confidence, "earlier": c1, "later": c2,
                 "change": c2 - c1, "later_antecedent_sessions": n2,
                 "found_in_earlier_mining": (r.antecedents, r.consequents) in rule_keys(train_rules)})

deliverable_check = pd.DataFrame(rows).sort_values("change").reset_index(drop=True)
deliverable_check.round(3)

In [ ]:
# Robustness: does the conclusion depend on where the cutoff is placed?
rows = []
for cutoff in ["2025-03-01", "2025-04-01", "2025-05-01"]:
    cutoff = pd.Timestamp(cutoff)
    e = transactions[transactions["transaction_date"] < cutoff]
    l = transactions[transactions["transaction_date"] >= cutoff]
    rules_e = mine_rules(e["basket"].tolist())
    l_onehot = to_onehot(l["basket"].tolist())
    changes = np.array([measure(l_onehot, r.antecedents, r.consequents)[0] - r.confidence
                        for r in rules_e.itertuples()])
    rows.append({
        "cutoff": cutoff.date(), "earlier_sessions": len(e), "later_sessions": len(l),
        "rules_mined_earlier": len(rules_e),
        "rules_stable": int((np.abs(changes) <= LARGE_DROP).sum()),
        "mean_change": changes.mean(), "max_abs_change": np.abs(changes).max(),
    })
pd.DataFrame(rows).round(3)

- **The rules generalise.** Mining on the earlier period alone gives 21 rules, and all 21 keep their confidence in the later period
  (changes between -0.055 and +0.127, none beyond the 0.15 limit). All 21 also keep a lift above 2.0 (the lowest is 2.26).
- **The two "differences" are not real differences.** 19 of the 21 rules match Member 3's set exactly. The other two are
  the same pairs (Space Frontier / Galactic Wars and Laugh Track / Office Antics) with the direction flipped,
  because the two directions are nearly tied in confidence. This supports reading those pairs as symmetric.
- **No sign of overfitting.** Rules are selected because their confidence cleared 0.30 in the earlier period, so
  they are biased to look slightly better there than they really are, and we would expect later confidence to drift
  *down*. Instead it went up on average (+0.033) and 16 of 21 rules rose.
- **Four rules moved by more than 2 standard errors, all upward:** the two Crime Drama rules that involve Silent Witness Redux
  (+0.127 and +0.116), Parenting Unscripted → Toon Town (+0.073) and Time Anomaly → Space Frontier (+0.077). With 21 rules,
  about one such result is expected by chance, and the two Crime Drama rules overlap heavily (both hinge on Silent Witness Redux sessions), so they are not independent.
  This hints that Crime Drama co-viewing strengthened in Apr to Jun, but it should be monitored, not treated as established.
- **Both cross-genre rules held up:** True Crime Files → Crime City (0.577 to 0.627) and Parenting Unscripted → Toon Town
  (0.520 to 0.593), so those findings are not a quirk of one quarter.
- **Robust to the cutoff:** moving the cutoff to 1 March or 1 May still leaves 21 of 21 rules stable.

**Limitations.** The data covers only six months, so seasonality across a full year is untested. The 0.15 limit is a
convention and the three-show rules rest on only about 64 to 83 later sessions (standard error near 0.05).
Stability shows the co-viewing patterns are consistent over time; it does not show that a recommendation
based on them would change what people watch, and the rules describe association, not causation.

In [ ]:
readable_export = readable[["rule", "genres", "rule_type", "sessions", "support", "confidence", "lift"]]
readable_export.to_csv("../outputs/readable_association_rules.csv", index=False)
validation.to_csv("../outputs/temporal_validation_results.csv", index=False)
print("Saved: readable_association_rules.csv, temporal_validation_results.csv")

## Role 4 Summary

The 21 cleaned rules from Member 3 were translated from show IDs into show names and genres using Member 1's lookup
table. Nineteen of the rules stay within a single genre and mostly restate the catalogue, and they collapse into 15 distinct
relationships because each of the three show trios yields three rules. The two cross-genre rules are the most interesting findings:
**True Crime Files - Crime City** (confidence 0.60 vs a 0.225 baseline) and **Parenting Unscripted - Toon Town** (0.557 vs 0.223).
They show viewers grouping shows differently from the catalogue labels.

For temporal validation, sessions were split by `transaction_date` at 1 April 2025 into an earlier period (1,778 sessions)
and a later period (1,722 sessions). Rules were mined on the earlier period alone with the same thresholds
(`min_support` 0.02, confidence 0.30, lift 2.0) and rechecked on the later period. All 21 rules stayed within 0.15 of their earlier
confidence (mean change +0.033), all kept a lift above 2.0, and the result was unchanged for cutoffs on 1 March and 1 May.
The rules therefore reflect stable co-viewing behaviour rather than an artefact of one time window.

## Role 5: Recommender Function and Persistence

This section turns the 21 cleaned rules into something that can be used: a function that takes a basket of
shows a viewer has watched, finds **every** rule that applies, ranks them by confidence and returns the
answer in show names. The rules and lookup tables are then saved so the recommender can be reloaded
without re-running the earlier stages, and a demo is run on baskets that were typed in by hand (not taken from the dataset).

**Design choices**

- A rule *matches* a basket when its whole antecedent is contained in the basket (`antecedent ⊆ basket`).
  A viewer who watched Crime City, The Detective and Toon Town still matches a rule whose antecedent is only Crime City + The Detective.
- Matching rules are ranked by **confidence** (as the lab specifies), with lift and then support as tie-breakers.
- Rules are mined on `show_id`, so the function accepts show names or IDs and reports names.
- Some matching rules recommend shows the viewer has already watched. These are kept by default (the guide asks for *all*
  matching rules) and flagged in an `already_watched` column, with the shows that would actually be new listed in `new_shows`. `only_new=True` removes the fully-watched ones.

### 1. Load the cleaned rules and the lookup table

In [ ]:
import json
import pickle
import datetime
import pandas as pd

rules_clean = pd.read_pickle("../outputs/clean_association_rules.pkl")   # Member 3's 21 rules (frozensets of show_ids)
show_lookup = pd.read_csv("../outputs/show_lookup.csv")                  # Member 1's lookup table
N_SESSIONS = len(pd.read_csv("../outputs/session_transactions.csv"))

print("Rules loaded:", len(rules_clean))
print("Shows in lookup:", len(show_lookup))
print("Sessions the rules were mined from:", N_SESSIONS)

### 2. Lookup structures

Plain dictionaries keyed by `show_id`. These are small, easy to save, and are all the recommender needs to translate between IDs and names.

In [ ]:
indexed = show_lookup.set_index("show_id")
name_of    = indexed["item_name"].to_dict()                          # S01 -> "Crime City"
genre_of   = indexed["category"].to_dict()                           # S01 -> "Crime Drama"
runtime_of = indexed["avg_episode_minutes"].astype(int).to_dict()    # S01 -> 45

assert len(name_of) == show_lookup["show_id"].nunique() == show_lookup["item_name"].nunique(), \
    "show_id and item_name should map one-to-one"

# Sanity check: every show that appears in a rule must exist in the lookup
shows_in_rules = set().union(*rules_clean["antecedents"], *rules_clean["consequents"])
assert shows_in_rules <= set(name_of), "A rule refers to a show that is missing from the lookup"
print(f"{len(name_of)} shows in the lookup; {len(shows_in_rules)} of them appear in at least one rule.")

### 3. The recommender function

`recommend()` does four things: it converts the basket to show IDs (rejecting anything unknown rather than silently ignoring it),
selects every rule whose antecedent is contained in the basket, ranks them, and returns names.

`watch_next()` is a small companion that collapses the rule-level output into one line per suggested show, because
several rules can point to the same show and a viewer only wants to see it once.

In [ ]:
def _label(show_ids, show_names):
    """frozenset of show_ids -> 'Show A + Show B' (sorted, so output is stable)."""
    return " + ".join(sorted(show_names[s] for s in show_ids))


def _resolve_basket(basket, show_names):
    """Accept show names or show_ids in any letter case; return a set of show_ids. Unknown entries raise an error."""
    name_to_id = {n.lower(): sid for sid, n in show_names.items()}
    watched, unknown = set(), []
    for item in basket:
        key = str(item).strip()
        if key.upper() in show_names:            # already a show_id, e.g. "s01"
            watched.add(key.upper())
        elif key.lower() in name_to_id:          # a show name, e.g. "crime city"
            watched.add(name_to_id[key.lower()])
        else:
            unknown.append(item)
    if unknown:
        raise ValueError(f"Unknown show(s): {unknown}. Valid shows: {sorted(show_names.values())}")
    return watched


def recommend(basket, rules, show_names, only_new=False, top_n=None):
    """
    Return every rule that applies to `basket`, ranked by confidence, as show names.

    basket      : list of show names or show_ids the viewer has watched
    rules       : DataFrame with frozenset columns 'antecedents' / 'consequents' and 'support', 'confidence', 'lift'
    show_names  : dict show_id -> show name
    only_new    : if True, drop rules whose consequent shows have all been watched already
    top_n       : optionally keep only the first n rules after ranking

    A rule matches when its whole antecedent is contained in the basket.
    """
    watched = _resolve_basket(basket, show_names)

    matched = rules[rules["antecedents"].apply(lambda a: a <= watched)].copy()
    matched["new_ids"] = matched["consequents"].apply(lambda c: c - watched)   # consequent shows not yet watched
    matched["already_watched"] = matched["new_ids"].apply(len) == 0
    if only_new:
        matched = matched[~matched["already_watched"]]

    matched = matched.sort_values(["confidence", "lift", "support"], ascending=False, kind="mergesort")
    if top_n is not None:
        matched = matched.head(top_n)

    out = pd.DataFrame({
        "if_watched":     matched["antecedents"].apply(_label, show_names=show_names),
        "then_recommend": matched["consequents"].apply(_label, show_names=show_names),
        "new_shows":      matched["new_ids"].apply(_label, show_names=show_names),
        "confidence":     matched["confidence"],
        "lift":           matched["lift"],
        "support":        matched["support"],
        "already_watched": matched["already_watched"],
    }).reset_index(drop=True)
    out.insert(0, "rank", range(1, len(out) + 1))
    return out


def watch_next(recs):
    """
    One line per show the viewer has not seen, from the highest-confidence rule that recommends it.
    Expects the output of recommend(). If a rule's consequent has several shows, its confidence is
    the confidence of the whole group, so it is only an approximation for each show inside it.
    """
    new = recs[~recs["already_watched"]].copy()
    new["show"] = new["new_shows"].str.split(" + ", regex=False)
    new = new.explode("show")                                   # one row per new show
    return (new.drop_duplicates("show", keep="first")           # recs is already sorted by confidence
               [["show", "confidence", "lift", "if_watched"]]
               .reset_index(drop=True))

### 4. Checks before saving

Two properties are tested on every rule: (a) feeding a rule's own antecedent to the recommender must return that rule, and
(b) the results must always be sorted by confidence, highest first.

In [ ]:
for r in rules_clean.itertuples():
    basket = [name_of[s] for s in r.antecedents]
    recs = recommend(basket, rules_clean, name_of)

    own_rule_found = ((recs["if_watched"] == _label(r.antecedents, name_of)) &
                      (recs["then_recommend"] == _label(r.consequents, name_of))).any()
    assert own_rule_found, f"Rule not returned for its own antecedent: {basket}"
    assert recs["confidence"].is_monotonic_decreasing, f"Not ranked by confidence for {basket}"

# Names, IDs and mixed case should all give the same answer
a = recommend(["Crime City", "The Detective"], rules_clean, name_of)
b = recommend(["s01", "S02"], rules_clean, name_of)
c = recommend(["  crime city ", "THE DETECTIVE"], rules_clean, name_of)
assert a.equals(b) and a.equals(c)

print(f"All {len(rules_clean)} rules are returned for their own antecedent, and results are always ranked by confidence.")
print("Names, IDs and mixed-case input give identical results.")

### 5. Persistence

Two copies are saved so the recommender can be used without re-running Roles 1 to 4:

- **`recommender_bundle.pkl`**: the rules (with their `frozenset` columns intact), all three lookup dictionaries, and the parameters used to mine them. Fast to reload, but pickle files are tied to the Python/pandas versions and should only be loaded from a trusted source.
- **`recommender_rules.json`**: the same rules and lookups in a plain, version-independent format that other tools can read. Sets are stored as sorted lists and rebuilt as `frozenset`s on load.

`show_lookup.csv` (Role 1) and `clean_association_rules.pkl/.csv` (Role 3) are already saved and are left unchanged.

In [ ]:
BUNDLE_PATH = "../outputs/recommender_bundle.pkl"
JSON_PATH   = "../outputs/recommender_rules.json"

PARAMS = {
    "min_support": 0.02,
    "min_confidence": 0.30,
    "min_lift": 2.00,
    "n_sessions": N_SESSIONS,
    "n_rules": len(rules_clean),
    "saved_at": datetime.datetime.now().isoformat(timespec="seconds"),
}

rule_cols = [c for c in ["antecedents", "consequents", "antecedent support", "consequent support",
                         "support", "confidence", "lift"] if c in rules_clean.columns]

# --- Pickle bundle ---
bundle = {
    "rules": rules_clean[rule_cols].reset_index(drop=True),
    "show_names": name_of,
    "show_genres": genre_of,
    "show_runtimes": runtime_of,
    "params": PARAMS,
}
with open(BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

# --- JSON copy ---
json_payload = {
    "params": PARAMS,
    "shows": {sid: {"name": name_of[sid], "category": genre_of[sid], "avg_episode_minutes": runtime_of[sid]}
              for sid in sorted(name_of)},
    "rules": [
        {
            "antecedents": sorted(r.antecedents),
            "consequents": sorted(r.consequents),
            "support": float(r.support),
            "confidence": float(r.confidence),
            "lift": float(r.lift),
        }
        for r in rules_clean.itertuples()
    ],
}
with open(JSON_PATH, "w") as f:
    json.dump(json_payload, f, indent=2)

print("Saved:", BUNDLE_PATH)
print("Saved:", JSON_PATH)

### 6. Reload and verify

The saved files are loaded back **as if in a fresh session**, and the recommender is run on the reloaded copies. Both copies must give exactly the same answers as the in-memory rules.

In [ ]:
# Reload the pickle bundle
with open(BUNDLE_PATH, "rb") as f:
    loaded = pickle.load(f)
rules_loaded, names_loaded = loaded["rules"], loaded["show_names"]

# Reload the JSON copy and rebuild the frozensets
with open(JSON_PATH) as f:
    payload = json.load(f)
rules_from_json = pd.DataFrame(payload["rules"])
rules_from_json["antecedents"] = rules_from_json["antecedents"].apply(frozenset)
rules_from_json["consequents"] = rules_from_json["consequents"].apply(frozenset)
names_from_json = {sid: info["name"] for sid, info in payload["shows"].items()}

# Same rules, same lookups
key = lambda df: {(r.antecedents, r.consequents) for r in df.itertuples()}
assert key(rules_loaded) == key(rules_clean) == key(rules_from_json)
assert names_loaded == name_of == names_from_json

# Same recommendations for every rule's antecedent, from all three sources
for r in rules_clean.itertuples():
    basket = [name_of[s] for s in r.antecedents]
    original = recommend(basket, rules_clean, name_of)
    pd.testing.assert_frame_equal(original, recommend(basket, rules_loaded, names_loaded))
    pd.testing.assert_frame_equal(original, recommend(basket, rules_from_json, names_from_json))

print("Reloaded pickle and JSON both reproduce the original recommendations.")
print("Saved parameters:", loaded["params"])

### 7. Demo on new, manually entered baskets

Everything below uses only the **reloaded** rules and lookup, and the baskets are typed in by hand, not taken from the dataset.

**Basket A** is a viewer who has watched two crime dramas.

In [ ]:
basket_a = ["Crime City", "The Detective"]
print("Basket:", basket_a)
print()
print("All matching rules, ranked by confidence:")
display(recommend(basket_a, rules_loaded, names_loaded).round(3))

In [ ]:
print("Same basket, showing only shows not yet watched (only_new=True):")
display(recommend(basket_a, rules_loaded, names_loaded, only_new=True).round(3))

print("Collapsed to one line per suggested show:")
display(watch_next(recommend(basket_a, rules_loaded, names_loaded)).round(3))

**Basket B** has a single documentary. This is the case that reaches across genres: True Crime Files is filed under Documentary, but the rules link it to Crime Drama.

In [ ]:
basket_b = ["True Crime Files"]
print("Basket:", basket_b)
display(recommend(basket_b, rules_loaded, names_loaded).round(3))

**Basket C** mixes genres: a viewer who has watched a Kids show and two Reality TV shows. Only rules whose entire antecedent fits inside the basket are returned.

In [ ]:
basket_c = ["Toon Town", "Parenting Unscripted", "Cooking Masters"]
print("Basket:", basket_c)
display(recommend(basket_c, rules_loaded, names_loaded).round(3))

### 8. Edge cases

A recommender that only works on friendly input is not finished. Three cases are checked: a basket that matches no rule, an empty basket, and a show that does not exist.

In [ ]:
# A show that never appears in any rule's antecedent cannot trigger a recommendation on its own
antecedent_shows = set().union(*rules_loaded["antecedents"])
silent_shows = [name_of[s] for s in sorted(name_of) if s not in antecedent_shows]

if silent_shows:
    print("Basket with no matching rule:", silent_shows[:1])
    out = recommend(silent_shows[:1], rules_loaded, names_loaded)
    print("Rules returned:", len(out))
else:
    print("Every show appears in at least one antecedent, so a single-show basket always matches something.")

print()
print("Empty basket -> rules returned:", len(recommend([], rules_loaded, names_loaded)))

try:
    recommend(["Crime City", "Breaking Bad"], rules_loaded, names_loaded)
except ValueError as e:
    print("\nUnknown show is rejected with a clear message:\n ", e)

## Role 5 Summary

The 21 cleaned rules were turned into a working recommender. `recommend()` accepts show names or IDs, returns **every** rule whose antecedent is contained in the
basket, ranks them by confidence (lift and support break ties), and reports the answer in show names. Rules whose consequent has already been watched are
flagged rather than hidden, and `only_new=True` and `watch_next()` give a cleaner "what to watch next" view. Unknown shows are rejected with an error instead of being silently ignored, and an empty or
non-matching basket returns an empty result.

The recommender was checked by confirming that every rule is returned for its own antecedent and that results are always sorted by confidence.

The rules, the three lookup dictionaries (name, genre, runtime) and the mining parameters were saved to `recommender_bundle.pkl`, with a version-independent copy in
`recommender_rules.json`. Both were reloaded and produced identical recommendations to the in-memory rules, and the demo baskets were run on the reloaded copies only.

**Limits to carry into the report (Member 6).** The recommender only reproduces co-viewing patterns from six months of data, so it cannot recommend a show the rules never mention, and it has nothing to say about a viewer whose basket matches no rule.
Rules are associations, not causes: a recommendation is a suggestion that has co-occurred, and whether it changes what people watch would need an A/B test. The rankings use confidence, as the guide requires, which favours popular consequents such as Crime City and Toon Town,
so lift should be read next to it.